# Representational compression of narrative details

This notebook answers one central question: **as a story is compressed into an age-steered summary, are narratively central details preserved or reweighted differently as a function of source age and target condition?**

The analysis deliberately separates four quantities:

1. **Whole-text displacement** — distance from the original-story representation to its summary representation.
2. **Grounding** — for every summary detail, its nearest original detail. This guarantees that every generated detail is accounted for.
3. **Coverage** — for every original detail, similarity to its nearest summary detail. This measures degree of preservation without imposing an arbitrary threshold.
4. **Relative weight change** — change in similarity between an original detail and the whole-text representation, from original story to summary.

Nearest-neighbor matching is the primary threshold-free analysis. Unique Hungarian matching is retained as a robustness check. Mean-pooled representations are a control for final-token representations.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from IPython.display import display

sns.set_theme(style='whitegrid', context='notebook')
ROOT = Path(os.environ.get('STORY_FEATURES_ROOT', Path.cwd())).resolve()
if not (ROOT / 'mechanistic_interpretability').exists():
    ROOT = ROOT.parent
MODEL = os.environ.get('REPRESENTATION_MODEL', 'gpt_oss_20b')
ANALYSIS_DIR = Path(os.environ.get(
    'REPRESENTATION_ANALYSIS_DIR',
    ROOT / 'mechanistic_interpretability/outputs/02_representation_extraction/analysis' / MODEL,
))
print('repository:', ROOT)
print('model:', MODEL)
print('analysis:', ANALYSIS_DIR)

## Load compact HPC results

The raw hidden states remain on HPC. The notebook only needs four compact tables produced by the analysis commands in the final section.

In [ ]:
paths = {
    'whole': ANALYSIS_DIR / 'story_summary_global_geometry.csv',
    'story': ANALYSIS_DIR / 'story_representation_compression_metrics.csv',
    'source_details': ANALYSIS_DIR / 'original_detail_representation_coverage.csv.gz',
    'summary_details': ANALYSIS_DIR / 'summary_detail_representation_alignment.csv.gz',
}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Run the HPC reduction commands first. Missing:\n' + '\n'.join(missing))
whole = pd.read_csv(paths['whole'], low_memory=False)
story = pd.read_csv(paths['story'], low_memory=False)
source_details = pd.read_csv(paths['source_details'], low_memory=False)
summary_details = pd.read_csv(paths['summary_details'], low_memory=False)
for frame in [whole, story, source_details, summary_details]:
    for column in ['source_age', 'target_age']:
        if column in frame:
            frame[column] = pd.to_numeric(frame[column], errors='coerce')
    if {'source_age', 'target_age'} <= set(frame):
        frame['age_shift'] = frame['target_age'] - frame['source_age']
print({name: len(frame) for name, frame in {
    'whole': whole, 'story': story, 'source_details': source_details, 'summary_details': summary_details
}.items()})

## Cohort audit

Before hypothesis testing, verify the number of stories, summaries, conditions, ages, and represented layers. Younger summaries are expected to be less frequent, so uncertainty—not raw means alone—must guide interpretation.

In [ ]:
audit = (story.groupby('condition', dropna=False)
         .agg(stories=('story_key', 'nunique'), summaries=('record_id', 'nunique'),
              source_age_min=('source_age', 'min'), source_age_max=('source_age', 'max'),
              target_age_min=('target_age', 'min'), target_age_max=('target_age', 'max'))
         .reset_index())
display(audit)
display(pd.crosstab(story['source_age'], story['condition']))

## Q1 — Does the whole representation move differently under age steering?

`eos_cosine` measures similarity between the one-pass original and summary representations. `mean_control_cosine` is the same-model mean-pooling control. Layerwise divergence indicates where the steering-conditioned compression geometry emerges.

In [ ]:
whole_profile = (whole.groupby(['layer', 'condition'], as_index=False)
                 .agg(stories=('story_key', 'nunique'), eos_cosine=('eos_cosine', 'mean'),
                      mean_control_cosine=('mean_control_cosine', 'mean')))
display(whole_profile)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(data=whole_profile, x='layer', y='eos_cosine', hue='condition', marker='o', ax=axes[0])
sns.lineplot(data=whole_profile, x='layer', y='mean_control_cosine', hue='condition', marker='o', ax=axes[1])
axes[0].set_title('Final-token whole-text similarity')
axes[1].set_title('Mean-pooling control')
plt.tight_layout()

## Q2 — Are central original details preferentially preserved?

For each original detail, `coverage_cosine` is its best similarity to any summary detail. `selected_by_any_summary_detail` asks whether at least one summary detail chose it as its nearest source. Neither measure uses a fixed cosine threshold.

In [ ]:
centrality_bins = pd.qcut(source_details['original_centrality_percentile'], 10, duplicates='drop')
coverage_profile = (source_details.assign(centrality_decile=centrality_bins)
                    .groupby(['condition', 'centrality_decile'], observed=True, as_index=False)
                    .agg(mean_coverage=('coverage_cosine', 'mean'),
                         selected_rate=('selected_by_any_summary_detail', 'mean'),
                         details=('original_detail_index', 'size')))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.lineplot(data=coverage_profile, x='centrality_decile', y='mean_coverage', hue='condition', marker='o', ax=axes[0])
sns.lineplot(data=coverage_profile, x='centrality_decile', y='selected_rate', hue='condition', marker='o', ax=axes[1])
axes[0].set_title('Semantic coverage by original centrality')
axes[1].set_title('Nearest-neighbor selection by original centrality')
for ax in axes:
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()

## Q3 — Does a detail gain or lose weight in the compressed whole?

For every original detail, relative weight change is

`cos(original detail, whole summary) − cos(original detail, whole original)`.

Positive values indicate that the detail becomes more aligned with the compressed representation; negative values indicate representational downweighting.

In [ ]:
weight_profile = (source_details.groupby(['story_key', 'record_id', 'condition', 'source_age', 'target_age'], as_index=False)
                  .agg(mean_weight_change=('original_detail_global_weight_change', 'mean'),
                       centrality_weight_correlation=('original_centrality', lambda s: np.nan)))
correlations = (source_details.groupby(['story_key', 'record_id'], sort=False)
                .apply(lambda d: d['original_centrality'].corr(d['original_detail_global_weight_change']))
                .rename('centrality_weight_correlation').reset_index())
weight_profile = weight_profile.drop(columns='centrality_weight_correlation').merge(correlations, on=['story_key', 'record_id'])
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.regplot(data=weight_profile, x='source_age', y='mean_weight_change', scatter_kws={'alpha': .25}, ax=axes[0])
sns.boxplot(data=weight_profile, x='condition', y='centrality_weight_correlation', ax=axes[1])
axes[0].set_title('Average detail reweighting by source age')
axes[1].set_title('Do central details gain more representational weight?')
plt.tight_layout()

## Q4 — Does the age effect change across development?

These plots keep source age and steering condition visible. The inferential model uses story-clustered standard errors so multiple summaries and details from one story are not treated as independent participants. A nonlinear spline can be added after inspecting support across ages.

In [ ]:
outcomes = [
    'mean_coverage_cosine', 'proportion_original_details_selected',
    'centrality_coverage_correlation', 'mean_original_detail_global_weight_change',
]
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
for ax, outcome in zip(axes.flat, outcomes):
    sns.regplot(data=story, x='source_age', y=outcome, scatter=False, ax=ax, color='black')
    sns.scatterplot(data=story, x='source_age', y=outcome, hue='condition', alpha=.25, s=25, ax=ax)
    ax.set_title(outcome.replace('_', ' ').title())
plt.tight_layout()

models = {}
for outcome in outcomes:
    frame = story.dropna(subset=[outcome, 'source_age', 'condition']).copy()
    model = smf.ols(f'{outcome} ~ C(condition) * source_age + n_original_details + n_summary_details', data=frame)
    models[outcome] = model.fit(cov_type='cluster', cov_kwds={'groups': frame['story_key']})
    print('\n', outcome)
    display(models[outcome].summary2().tables[1])

## Q5 — Are results robust to matching and representation choices?

Three prespecified checks prevent the result from depending on one arbitrary choice:

- Compare nearest-neighbor `grounding_cosine` with `unique_match_cosine`.
- Repeat the HPC reducer using `--representation mean` as a pooling control.
- Repeat the complete notebook for Qwen3-8B and compare effect direction on the shared story set.

In [ ]:
robustness = (summary_details.groupby(['story_key', 'record_id', 'condition'], as_index=False)
              .agg(nearest_neighbor=('grounding_cosine', 'mean'), unique_matching=('unique_match_cosine', 'mean')))
display(robustness.groupby('condition')[['nearest_neighbor', 'unique_matching']].agg(['count', 'mean', 'std']))
sns.scatterplot(data=robustness, x='nearest_neighbor', y='unique_matching', hue='condition', alpha=.4)
limits = [robustness[['nearest_neighbor', 'unique_matching']].min().min(),
          robustness[['nearest_neighbor', 'unique_matching']].max().max()]
plt.plot(limits, limits, '--', color='black')
plt.title('Nearest-neighbor versus unique matching')
plt.show()

## Q6 — Does narrative context change how a detail is represented?

For GPT-OSS, the isolated-detail control encodes the same wording without its story. `contextualization_shift = 1 − cosine(contextual, isolated)` measures how strongly narrative context changes that detail. This is not deletion influence: it directly compares the same detail with and without surrounding context.

In [ ]:
context_data = source_details.dropna(subset=['contextualization_shift']).copy()
if context_data.empty:
    print('No isolated-detail controls were supplied to the HPC reducer.')
else:
    context_story = (context_data.groupby(['story_key', 'record_id', 'condition', 'source_age'], as_index=False)
                     .agg(mean_contextualization_shift=('contextualization_shift', 'mean'),
                          centrality_context_correlation=('original_centrality',
                              lambda s: np.nan)))
    correlations = (context_data.groupby(['story_key', 'record_id'], sort=False)
                    .apply(lambda d: d['original_centrality'].corr(d['contextualization_shift']))
                    .rename('centrality_context_correlation').reset_index())
    context_story = context_story.drop(columns='centrality_context_correlation').merge(correlations, on=['story_key', 'record_id'])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.regplot(data=context_story, x='source_age', y='mean_contextualization_shift', ax=axes[0])
    sns.boxplot(data=context_story, x='condition', y='centrality_context_correlation', ax=axes[1])
    axes[0].set_title('Narrative contextualization by source age')
    axes[1].set_title('Does context reshape central details differently?')
    plt.tight_layout()

## Interpretation checklist

A defensible positive result requires convergence across levels:

1. Whole-story displacement differs by age condition.
2. Original centrality predicts continuous coverage or threshold-free selection.
3. Central details show systematically different whole-summary weight changes.
4. The condition × source-age interaction indicates whether the steering effect changes developmentally.
5. Effect direction survives mean pooling, unique matching, and Qwen replication.

Do not interpret a condition mean without checking the source-age distribution and story-level uncertainty.